[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/boltzgen/advanced/10_small_molecule/10_small_molecule_lab.ipynb)


# 10 — 소분자 결합 + 친화도 예측 실습

> 본문 [`10_small_molecule.md`](10_small_molecule.md) 와 **한 절씩 짝지어** 보세요.
> 이 노트북의 표·그래프·수치는 **여러분이 직접 돌린 결과**(`my_run/`)에서 계산합니다.
> **① 직접 설계 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 갑니다. 설계 셀은 NVIDIA GPU 전용이에요(CPU 폴백 없음) — Colab 이면 **런타임 → 런타임 유형 변경 → T4 GPU**.

## 0) 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `10_small_molecule/` 로 이동합니다. 로컬에서 `10_small_molecule/` 안에 열었다면 클론 없이 진행돼요.

In [ ]:
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # fork 했다면 본인 주소로 바꾸세요
CLONE_AS = "bio-guides"
CHAPTER  = "10_small_molecule"
PIP_PKGS = "pandas matplotlib gemmi py3Dmol"   # 없으면 설치할 분석 라이브러리

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

# HF 가중치 다운로드가 멈춘 채 매달리지 않도록 타임아웃을 겁니다.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, quiet=False):
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        subprocess.run(cmd, shell=True, check=True); return
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        raise subprocess.CalledProcessError(p.returncode, cmd)

_MARK = "boltzgen_viz.py"          # 이 파일이 있는 폴더가 advanced/ 루트

def _find_root():
    """advanced/ 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):   # 클론 직후: cwd 아래로만 깊이 3까지
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "advanced/ 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)          # 이 챕터 폴더로 이동 → data/ 상대경로가 그대로 동작
sys.path.insert(0, str(ADV_ROOT))     # boltzgen_viz import 보장
import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update", quiet=True)
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y fonts-nanum", quiet=True)

# import 안 되는 패키지만 설치합니다.
import importlib, importlib.util
_IMPORT_NAME = {"scikit-learn": "sklearn", "pillow": "PIL", "biopython": "Bio"}
def _have(mod):
    try: return importlib.util.find_spec(mod) is not None
    except Exception: return False
def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p))]
    if miss:
        print("필요 라이브러리 설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))  # python -m pip (Ch.03 권고)
        importlib.invalidate_caches()
if PIP_PKGS:
    _ensure(PIP_PKGS)

# 내 결과는 my_run/ 에 쌓이고, 없으면 커밋된 레퍼런스로 폴백합니다.
MY  = pathlib.Path("my_run")
MY.mkdir(exist_ok=True)

def find_one(pattern, ref, quiet=False):
    """my_run/ 에서 먼저 찾고, 없으면 레퍼런스 폴더에서 찾습니다."""
    for base in (MY/"final_ranked_designs", MY/"intermediate_designs_inverse_folded", MY):
        hits = sorted(base.glob(pattern))
        if hits:
            if not quiet: print(f"[내 결과]   {hits[0]}")
            return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"{ref} 에서 '{pattern}' 을 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def cols_in(df, *names):
    """내 실행 결과와 레퍼런스는 컬럼 구성이 조금 다를 수 있어, 있는 컬럼만 고른다."""
    missing = [c for c in names if c not in df.columns]
    if missing:
        print("(이 실행에는 없는 컬럼 — 건너뜁니다:", ", ".join(missing) + ")")
    return [c for c in names if c in df.columns]

def inherit_run(*rel_paths):
    """앞 챕터에서 돌린 설계 결과를 이어받습니다(없으면 레퍼런스로 폴백)."""
    global MY
    if (MY / "final_ranked_designs").exists():
        return MY
    for rel in rel_paths:
        p = pathlib.Path(rel)
        if (p / "final_ranked_designs").exists():
            print("[이어받기] 앞 챕터에서 직접 돌린 결과를 사용합니다:", p)
            MY = p
            return MY
    return MY

# 레퍼런스 그림을 덮어쓰지 않도록 my_ 접두어.
def my_fig(name):
    return f"my_{name}"

print("작업 폴더 :", pathlib.Path.cwd())

## 1) 직접 돌려보기 — 내 결과 만들기

- 학습용 규모 `num_designs=8 --budget=4` (레퍼런스 결과는 num_designs=30)
- 소요 시간 실측 **585초**(최종 4개) — **24GB GPU · 가중치 캐시** 기준이에요. Colab T4 는 가속 커널이 꺼져 더 걸리고(T4 실측치 없음), 첫 실행은 가중치 ~6GB 다운로드가 더 붙어요.
- 건너뛰면 아래 분석이 커밋된 레퍼런스 결과로 이어집니다.

In [ ]:
SPEC, PROTOCOL = "example/protein_binding_small_molecule/chorismite.yaml", "protein-small_molecule"
NUM_DESIGNS, BUDGET = 8, 4

import shutil
OUT = MY.resolve()

def _gpu():
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        return shutil.which("nvidia-smi") is not None

if not _gpu():
    print("GPU 런타임이 아니라 설계 실행을 건너뜁니다 — 아래 분석은 레퍼런스 결과로 이어집니다.")
    print("  · 직접 돌리려면 Colab [런타임 → 런타임 유형 변경 → T4 GPU] 후 이 셀을 다시 실행하세요.")
else:
    SRC = ADV_ROOT / ".boltzgen_src"          # 예제 명세·타깃 CIF 가 들어 있는 BoltzGen 소스
    if not SRC.exists():
        _run(f'git clone --depth 1 https://github.com/HannesStark/boltzgen.git "{SRC}"')
    if not _have("boltzgen"):
        _run(f'"{sys.executable}" -m pip -q install -e "{SRC}"')
    try:
        _run(f'cd "{SRC}" && boltzgen run {SPEC} --output "{OUT}" --protocol {PROTOCOL} '
             f'--num_designs {NUM_DESIGNS} --budget {BUDGET}')
        print("\n내 결과 →", OUT / "final_ranked_designs")
    except Exception as e:
        print("\n설계 실행이 도중에 멈췄어요 —", type(e).__name__)
        print("  · 이 셀을 다시 실행하면 같은 --output 산출물을 재사용해 이어서 끝냅니다(실측 763초 → 재개 486초).")
        print("  · GPU 메모리가 부족했다면 NUM_DESIGNS 4, BUDGET 2 로 줄이세요.")

## 2) 리간드를 어떻게 적어 넣나 — CCD vs SMILES (본문 10.2)

방금 돌린 명세에서 타깃 자리를 차지한 건 `ligand` 한 줄이에요. 소분자는 두 가지로 적습니다.

```yaml
# 방법 1 — CCD 코드(PDB 화학성분 사전). 결정구조에 이미 등장한 분자
- ligand: { id: B, ccd: TSA }

# 방법 2 — SMILES. 표준 코드가 없는 신약 후보 등
- ligand: { id: B, smiles: "C1CNC[C@@H]1OC2=C(...)..." }
```

`chorismite.yaml` 은 첫 번째 방식이고 `TSA` 는 chorismate 전이상태 유사체예요. RCSB 에서 코드가 검색되면 `ccd`, 안 나오면 `smiles` 나 `.sdf` 로 넣습니다. 무엇이 실제로 들어갔는지는 결과 구조 파일이 알려줘요.

In [ ]:
cif = find_one("final*designs/*.cif", "data/small_molecule")
het = [l.split() for l in pathlib.Path(cif).read_text().splitlines() if l.startswith("HETATM")]
codes = sorted({f[5] for f in het})
print("결과 구조에 들어 있는 비폴리머 코드", codes)
for code in codes:
    names = [f[3] for f in het if f[5] == code]
    print(f"  {code} — 원자 {len(names)}개 | {' '.join(names)}")

리간드는 **수소를 뺀 heavy atom 목록**으로 들어가요. TSA 는 16개고, 이 16이 바로 다음 절의 토큰 계산에 그대로 나타납니다.

## 3) `protein-small_molecule` — affinity 스텝이 붙은 프로토콜 (본문 10.3)

리간드가 정해졌으면 프로토콜을 고를 차례예요. 소분자 전용 프로토콜에는 다른 프로토콜에 없는 **affinity 스텝**이 하나 더 있어요.

```bash
# 아래 분석이 쓰는 레퍼런스를 만든 명령
boltzgen run example/protein_binding_small_molecule/chorismite.yaml --output workbench/sm \
  --protocol protein-small_molecule --num_designs 30 --budget 10

# 프로덕션 규모 — 소분자는 결합면이 좁아 후보를 크게 잡습니다(수 시간~수십 시간)
boltzgen run example/protein_binding_small_molecule/chorismite.yaml --output workbench/sm \
  --protocol protein-small_molecule --num_designs 3000 --budget 40
```

디자인당 affinity 예측에만 ~17초가 더 붙어요. 그리고 **친화도 예측은 이 프로토콜에서만** 나옵니다 — 다른 프로토콜로 돌리면 `affinity_*` 컬럼이 아예 생기지 않아요. 스텝 구성은 산출물의 `steps.yaml` 에 그대로 남습니다.

In [ ]:
steps_txt = pathlib.Path(find_one("steps.yaml", "data/small_molecule")).read_text()
step_names = [l.split("name:")[1].strip() for l in steps_txt.splitlines() if "name:" in l]
print(f"스텝 {len(step_names)}개 —", " → ".join(step_names))
print("affinity 스텝 있음 — 아래 7절의 친화도 분석이 그대로 이어집니다." if "affinity" in step_names
      else "affinity 스텝 없음 — 이 산출물에는 affinity_* 컬럼이 없어 7절을 건너뛰게 됩니다.")

## 4) 복합체 토큰 세기 — 폴리머는 잔기, 소분자는 원자 (본문 10.3)

7스텝이 확인됐으니 이제 이 복합체가 모델에 얼마나 큰 입력인지 봅니다. BoltzGen 이 세는 단위는 **토큰**이에요. 폴리머는 **잔기당 1토큰**, 소분자는 **원자당 1토큰**이라 `num_prot_tokens + num_lig_atoms = num_tokens` 가 딱 맞아떨어져야 해요.

In [ ]:
from boltzgen_viz import load_metrics, metrics_overview
import pandas as pd
CSV = str(find_one("final_designs_metrics_*.csv", "data/small_molecule"))
df = pd.read_csv(CSV).sort_values("final_rank")

tok = cols_in(df, "final_rank", "id", "num_prot_tokens", "num_lig_atoms", "num_tokens")
display(df[tok])
if {"num_prot_tokens", "num_lig_atoms", "num_tokens"} <= set(df.columns):
    ok = int((df.num_prot_tokens + df.num_lig_atoms == df.num_tokens).sum())
    print(f"num_prot_tokens + num_lig_atoms = num_tokens 성립 {ok}/{len(df)}")
    print(f"단백질 {int(df.num_prot_tokens.min())}~{int(df.num_prot_tokens.max())}잔기"
          f" + 리간드 {int(df.num_lig_atoms.iloc[0])}원자"
          f" = 복합체 {int(df.num_tokens.min())}~{int(df.num_tokens.max())}토큰")

단백질 길이는 명세의 `sequence: 140..180` 안에서 실행마다 달라지고, 리간드 16은 고정이에요. GPU 메모리는 이 토큰 수에 좌우되니 OOM 이 나면 `sequence` 상한이나 `--num_designs` 를 줄이세요.

## 5) 최종 선별셋 — 한 표, 한 그림 (본문 10.6)

입력이 확인됐으니 결과를 봅니다. 4번째 패널이 다른 챕터의 산점도 대신 **예측 친화도 바**로 바뀌는 게 이 프로토콜의 표식이에요.

In [ ]:
df[cols_in(df, "id", "final_rank", "design_ptm", "design_to_target_iptm", "filter_rmsd",
           "plip_hbonds_refolded", "delta_sasa_refolded",
           "affinity_pred_value", "affinity_probability_binary", "num_design")]

In [ ]:
rows = load_metrics(CSV)
FIG = my_fig("10_small_molecule_metrics.png")
metrics_overview(rows, "Small-Molecule Binder (chorismate) — Design Metrics Overview",
                 FIG, panel4="affinity")
from IPython.display import Image; Image(FIG)

레퍼런스(num_designs=30) 기준으로 ipTM 0.723~0.841, RMSD 0.57~1.76Å 이 나와요. 소분자처럼 작고 조밀한 인터페이스는 ipTM 이 높게 나오는 경향이 있으니 **단백질-단백질 ipTM 과 절대값을 비교하지 마세요**(Ch.05). 같은 실행 안에서의 상대 순위로만 씁니다.

## 6) 포켓 품질 — 얼마나 묻히고, 몇 개나 붙잡나 (본문 10.4)

ipTM·RMSD 는 "구조를 믿을 수 있나"까지만 말해줘요. 소분자가 실제로 주머니에 묻혔는지는 따로 봐야 합니다. `delta_sasa_refolded`(묻힌 표면적, Å²)·`plip_hbonds_refolded`(수소결합)·`plip_saltbridge_refolded`(염다리)가 그 지표예요.

In [ ]:
pk = cols_in(df, "final_rank", "id", "delta_sasa_refolded", "plip_hbonds_refolded",
             "plip_saltbridge_refolded", "filter_rmsd")
sub = df[pk].sort_values("delta_sasa_refolded", ascending=False) if "delta_sasa_refolded" in df.columns else df[pk]
print("묻힌 표면적(ΔSASA) 큰 순 — 클수록 주머니에 깊이 파묻힌 거예요")
display(sub.reset_index(drop=True))
if {"delta_sasa_refolded", "filter_rmsd"} <= set(df.columns):
    a = df.loc[df.delta_sasa_refolded.idxmax()]
    b = df.loc[df.filter_rmsd.idxmin()]
    print(f"\nΔSASA 최대  rank{int(a.final_rank)} {a.id} — {a.delta_sasa_refolded:.1f} A^2, RMSD {a.filter_rmsd:.2f} A")
    print(f"RMSD 최저   rank{int(b.final_rank)} {b.id} — {b.delta_sasa_refolded:.1f} A^2, RMSD {b.filter_rmsd:.2f} A")

ΔSASA 가 가장 큰 디자인과 RMSD 가 가장 낮은 디자인은 서로 다른 후보로 나옵니다. 묻힘(ΔSASA·H-bond·염다리)은 **포켓이 소분자를 얼마나 감싸는가**, RMSD 는 **그 구조를 설계 서열이 재현하는가** — 서로 다른 축이라 하나로 다른 하나를 대신할 수 없어요.

판정은 각각 따로 하고 둘 다 만족하는 후보를 고릅니다. 레퍼런스 기준 ΔSASA 252~362Å², H-bond 1~9개, 염다리는 전부 0이라 이 포켓의 결합은 묻힘과 수소결합이 담당해요. ΔSASA 가 유독 작은 디자인은 리간드가 표면에 얹혀 있다는 뜻이니 후보에서 빼세요.

## 7) 친화도 컬럼 6종과 랭킹 (본문 10.5)

포켓이 잘 감싼다고 결합이 센 건 아니에요. 결합 세기는 affinity 스텝이 따로 예측해 줍니다.

| 컬럼 | 의미 |
|------|------|
| `affinity_pred_value` | 예측 결합 친화도(회귀값, 클수록 강한 결합) |
| `affinity_probability_binary` | "결합한다" 이진 확률 |
| `affinity_pred_value1`, `_value2` | 보조 회귀값 |
| `affinity_probability_binary1`, `_binary2` | 보조 이진 분류 확률 |
| `rank_affinity_probability_binary1` | 최종 랭킹에 들어가는 친화도 순위 |

In [ ]:
aff = cols_in(df, "final_rank", "id", "affinity_pred_value", "affinity_probability_binary",
              "affinity_pred_value1", "affinity_pred_value2",
              "affinity_probability_binary1", "affinity_probability_binary2",
              "rank_affinity_probability_binary1")
display(df[aff])
if "affinity_pred_value" in df.columns:
    k = min(3, len(df))
    print(f"최종 {len(df)}개 전체 기준 affinity_pred_value 상위 {k}")
    display(df.nlargest(k, "affinity_pred_value")[
        cols_in(df, "final_rank", "id", "affinity_pred_value", "affinity_probability_binary",
                "design_to_target_iptm", "filter_rmsd")].reset_index(drop=True))
if "rank_affinity_probability_binary1" in df.columns:
    t = df.nsmallest(1, "rank_affinity_probability_binary1").iloc[0]
    print(f"\n친화도 순위 1위는 {t.id}(final_rank {int(t.final_rank)}) — 최종 순위는 여러 rank_* 의 종합이에요.")

친화도로 뽑은 상위와 `final_rank` 상위는 일치하지 않아요. **최종 10개 전체**를 기준으로 보면 `affinity_pred_value` 는 rank 10·9·2 가 높고(2.71/2.49/2.39), 본문 10.6 표처럼 **상위 5개 안에서만** 보면 rank 2·3(2.39/2.34)이 가장 높습니다 — 어느 범위에서 고른 추천인지 늘 같이 말해야 해요.

판정 기준. 친화도는 **절대값이 아니라 같은 실행 안의 상대 순위**로 씁니다. 구조 지표(ipTM·RMSD)와 친화도가 함께 상위인 후보를 실험 후보로 올리고, 그 후보는 AutoDock Vina 재도킹·MD 로 독립 검증하세요(Ch.06.7).

```bash
vina --receptor protein.pdbqt --ligand ligand.pdbqt --out docked.pdbqt
```

## 8) 포켓을 눈으로 — rank 1 디자인 3D (본문 10.4)

6)절의 ΔSASA 는 "리간드가 얼마나 묻혔나"를 숫자 하나로 압축한 값이에요.
그 숫자가 무엇을 뜻하는지는 포켓을 직접 봐야 압니다.

**주황이 설계 단백질(cartoon), 초록 stick + 반투명 구가 리간드**예요.
리간드는 폴리머가 아니라 HETATM 으로 들어 있어 cartoon 으로는 안 그려지니 따로 칠합니다. 마우스로 회전·확대할 수 있어요(휠 = 줌, 드래그 = 회전). 구조가 안 보이면 `py3Dmol` 설치 로그를 확인하고 셀을 한 번 더 실행하세요.

In [ ]:
import importlib.util, sys, pathlib
for _pkg in ("py3Dmol", "gemmi"):                       # 없으면 그 자리에서 설치
    if importlib.util.find_spec(_pkg) is None:
        _run(f'"{sys.executable}" -m pip -q install {_pkg}')
import py3Dmol, gemmi, pandas as pd
from IPython.display import display

C_DESIGN  = ["#e8883a", "#c0392b"]              # 설계 사슬 — 주황·빨강
C_TARGET  = ["#3477b5", "#7f8c8d", "#8e44ad"]   # 타깃 단백질 — 파랑·회색·보라
C_NUCLEIC = "#1aa090"                           # 타깃 핵산 — 청록

def chain_seq3d(ch):
    """사슬의 한 글자 서열(폴리머 잔기만)."""
    poly = ch.get_polymer()
    return gemmi.one_letter_code([r.name for r in poly]).upper() if len(poly) else ""

def chain_kind3d(ch):
    """protein / dna / rna / hetero(리간드·금속) 판별."""
    poly = ch.get_polymer()
    if not len(poly):
        return "hetero"
    t = str(poly.check_polymer_type())
    return "dna" if "Dna" in t else "rna" if "Rna" in t else "protein"

def load_design(pattern, ref, row=None,
                seq_cols=("full_sequence_1", "full_sequence_2", "designed_chain_sequence")):
    """최종 CIF 를 찾아 열고 (pdb문자열, model, 설계사슬, 타깃사슬, 사슬→CSV컬럼) 을 돌려줍니다.
    설계 사슬은 CSV 의 설계 서열과 사슬 서열이 같은 것으로 정해요 — 사슬 라벨을 하드코딩하지 않습니다."""
    cif = find_one(pattern, ref)                        # [내 결과]/[레퍼런스] 를 스스로 출력합니다
    print("띄울 구조 :", cif, "→", "내 결과 my_run/" if "my_run" in str(cif) else "커밋된 레퍼런스 data/")
    st = gemmi.read_structure(str(cif)); st.setup_entities()
    model = st[0]

    want = {}
    if row is not None:
        for col in seq_cols:
            v = row[col] if col in getattr(row, "index", []) else None
            if isinstance(v, str) and v.strip():
                want.setdefault(v.strip().upper(), col)   # 같은 서열이면 먼저 온 컬럼을 씁니다
    match  = {ch.name: want[chain_seq3d(ch)] for ch in model if chain_seq3d(ch) in want}
    design = list(match)
    if not design:                                        # 폴백 — 설계물은 보통 가장 짧은 단백질 사슬
        prot = [ch for ch in model if chain_kind3d(ch) == "protein"]
        design = [min(prot, key=lambda c: len(c.get_polymer())).name] if prot else []
        print("  (CSV 서열과 일치하는 사슬이 없어 가장 짧은 단백질 사슬을 설계물로 봅니다)")
    target = [ch.name for ch in model if ch.name not in design]

    rows = []
    for ch in model:
        n = len(ch.get_polymer()) or len(ch)
        # hetero(리간드·금속)는 one-letter 서열이 안 나와 칸이 비어 보여요 — 잔기 이름을 대신 보여줍니다.
        what = chain_seq3d(ch)[:28] or "·".join(sorted({r.name for r in ch}))
        is_des = ch.name in design
        rows.append({"사슬": ch.name,
                     "역할": "설계 — 우리가 만든 것" if is_des else "타깃 — 붙일 상대",
                     "그림 색": "주황" if is_des else ("청록" if chain_kind3d(ch) in ("dna", "rna") else "파랑"),
                     "종류": chain_kind3d(ch),
                     "잔기 수": n,
                     "서열 (앞 28자)": what})
    display(pd.DataFrame(rows))
    return st.make_pdb_string(), model, design, target, match

def complex_view(pdb, model, design, target, width=760, height=540):
    """설계 사슬(주황)·타깃 단백질(파랑)·핵산(청록) cartoon + 비폴리머(리간드 stick·이온 sphere)."""
    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb, "pdb")
    view.setStyle({}, {"cartoon": {"color": "#cfd6dd"}})
    ti = 0
    for ch in model:
        kind = chain_kind3d(ch)
        if kind == "hetero":
            continue
        if ch.name in design:
            color = C_DESIGN[design.index(ch.name) % len(C_DESIGN)]
        elif kind in ("dna", "rna"):
            color = C_NUCLEIC
        else:
            color = C_TARGET[ti % len(C_TARGET)]; ti += 1
        style = {"cartoon": {"color": color}}
        if kind in ("dna", "rna"):
            style["stick"] = {"color": color, "radius": 0.12}   # 염기까지 보이게
        view.setStyle({"chain": ch.name}, style)
    for name, natoms in {r.name: len(r) for ch in model for r in ch if r.het_flag == "H"}.items():
        if natoms == 1:                                   # 금속·이온
            view.addStyle({"resn": name}, {"sphere": {"color": "#6c5ce7", "radius": 0.9}})
        else:                                             # 리간드
            view.addStyle({"resn": name}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.22}})
    view.setBackgroundColor("white")
    return view

def designed_segments(full, des, min_len=3):
    """이어붙여 저장된 재설계 구간(des)을 전체 서열(full) 위에 되짚어 (시작,끝) 1-based 목록으로.
    CDR 번호를 노트북에 적어 두지 않고 결과 CSV 에서 되찾기 위한 것입니다."""
    full, des = str(full).upper(), str(des).upper()
    out, i, pos = [], 0, 0
    while i < len(des):
        best, at = 0, -1
        for L in range(len(des) - i, min_len - 1, -1):
            j = full.find(des[i:i + L], pos)
            if j >= 0:
                best, at = L, j
                break
        if best == 0:
            i += 1
            continue
        out.append((at + 1, at + best))
        i += best; pos = at + best
    return out

def seg_resi(model, chain_id, segs):
    """서열 위 구간 → 그 사슬의 실제 잔기 번호 목록(3Dmol 의 resi 선택용)."""
    ch = {c.name: c for c in model}[chain_id]
    nums = [r.seqid.num for r in ch.get_polymer()]
    return [n for a, b in segs for n in nums[a - 1:b]]

# 사슬을 찾아준 컬럼 → 그 사슬의 '재설계 구간' 컬럼
DES_COL = {"full_sequence_1": "designed_sequence_1", "full_sequence_2": "designed_sequence_2",
           "designed_chain_sequence": "designed_sequence"}

top = df.sort_values("final_rank").iloc[0]
print("rank 1 디자인 :", top["id"])
pdb, model, DESIGN, TARGET, MATCH = load_design(
    f"final*designs/*{top['id']}*.cif", "data/small_molecule", row=top)

# 리간드 = 비폴리머(HETATM) 잔기. 코드·원자 수는 구조에서 읽습니다(2)절과 같은 값이어야 해요).
LIG = {r.name: len(r) for ch in model for r in ch if r.het_flag == "H" and len(r) > 1}
print("  리간드", ", ".join(f"{n}({k}원자)" for n, k in LIG.items()) or "없음")

view = complex_view(pdb, model, DESIGN, TARGET)
for name in LIG:
    view.addStyle({"resn": name}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.3}})
    view.addStyle({"resn": name}, {"sphere": {"colorscheme": "greenCarbon", "opacity": 0.35}})

# 포켓 잔기 — 리간드에서 4.5A 안에 있는 설계 단백질 잔기(거리로 찾습니다, 목록 하드코딩 없음)
lig_pos = [a.pos for ch in model for r in ch if r.name in LIG for a in r]
pocket = {}
for ch in model:
    if ch.name not in DESIGN:
        continue
    hit = [r.seqid.num for r in ch
           if any(a.pos.dist(p) <= 4.5 for a in r for p in lig_pos)]
    if hit:
        pocket[ch.name] = hit
        view.addStyle({"chain": ch.name, "resi": hit},
                      {"stick": {"colorscheme": "orangeCarbon", "radius": 0.14}})
        shown = "·".join(map(str, hit[:12])) + (f" … (앞 12개만, 전체 {len(hit)}개)" if len(hit) > 12 else "")
        print(f"  사슬 {ch.name} 포켓 잔기 {len(hit)}개 @ {shown}")

view.zoomTo({"resn": list(LIG)} if LIG else {})     # 포켓을 중심으로 당겨 봅니다
view.show()

print("\n무엇을 봐야 하나")
print("  · 초록 리간드가 주황 단백질 안쪽 주머니에 파묻혀 있으면 정상이에요 — 표면에 얹혀 있으면 ΔSASA 가 작게 나옵니다.")
print("  · 리간드를 둘러싼 주황 stick(포켓 잔기)이 사방을 감싸고 있는지 보세요. 한쪽만 닿아 있으면 결합이 약해요.")
print("  · 화면을 넓게 당기면(휠 축소) 단백질 전체에서 포켓 위치도 확인할 수 있어요.")
if {"delta_sasa_refolded", "plip_hbonds_refolded"} <= set(df.columns):
    print(f"\n이 디자인의 숫자 — ΔSASA {top['delta_sasa_refolded']:.1f} A^2 / "
          f"H-bond {int(top['plip_hbonds_refolded'])}개 (6)절 표와 같은 값)")

## 9) 이 챕터에서 확인한 것 (본문 10.6~10.7)

리간드를 `ccd`/`smiles` 로 적어 넣고(2절), affinity 스텝이 붙은 7스텝 프로토콜로 돌려(3절), 복합체가 단백질 잔기 + 리간드 원자로 토큰화되는 것을 확인했어요(4절). 결과는 구조 신뢰도(5절)·포켓 묻힘(6절)·예측 친화도(7절) **세 축을 따로** 읽고, 셋이 함께 상위인 후보만 실험으로 넘깁니다.

다음 챕터의 타깃은 단백질도 소분자도 아닌 **핵산**이에요. 리간드처럼 원자로 세지 않고 폴리머로 다루면서도, 온통 음전하인 골격 때문에 인터페이스 지표가 지금까지와 전혀 다른 값을 냅니다.